In [32]:
import jax
import jax.numpy as jnp
from unscented import UnscentedKalmanFilter

jax.config.update("jax_enable_x64", True)

import jax.scipy.linalg as jsp_linalg
import optax
from profilter import chol_softplus
from unscented import unscented_transform

# Testing Unscented transform

In [33]:
#Testing unscented transform with toy example
def unscented_transform(mean, cov, f, alpha=1e-3, beta=2.0, kappa=0.0):
    d = mean.shape[0]
    lam = alpha**2 * (d + kappa) - d
    print("lambda:", lam)

    S = jnp.linalg.cholesky((d + lam) * cov) #Matrix square root
    print("S:", S)

    sigma_points = jnp.concatenate([
        mean[None, :], #changes the shape of mean from (2,) into (1, 2)
        mean[None, :] + S.T,
        mean[None, :] - S.T,
    ], axis=0)
    print("sigma_points:", sigma_points)

    Wm = jnp.concatenate([ #creates an array of W(m) value from 0 to 2d
        jnp.array([lam / (d + lam)]),
        jnp.full(2 * d, 1.0 / (2 * (d + lam)))
    ])
    print("Wm:", Wm)

    Wc = Wm.at[0].add(1 - alpha**2 + beta)
    print("Wc:", Wc)

    transformed = jnp.stack([f(sp) for sp in sigma_points])
    print("transformed:", transformed)

    mean_pred = jnp.sum(Wm[:, None] * transformed, axis=0)
    print("mean_pred:", mean_pred)

    diff = transformed - mean_pred
    cov_pred = jnp.einsum('i,ij,ik->jk', Wc, diff, diff)
    print("cov_pred:", cov_pred)

    return mean_pred, cov_pred

mean = jnp.array([0.0, 0.0])
cov = jnp.eye(2)
unscented_transform(mean, cov, lambda x: x)

lambda: -1.999998
S: [[0.00141421 0.        ]
 [0.         0.00141421]]
sigma_points: [[ 0.          0.        ]
 [ 0.00141421  0.        ]
 [ 0.          0.00141421]
 [-0.00141421  0.        ]
 [ 0.         -0.00141421]]
Wm: [-999998.99997124  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
Wc: [-999995.99997224  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
transformed: [[ 0.          0.        ]
 [ 0.00141421  0.        ]
 [ 0.          0.00141421]
 [-0.00141421  0.        ]
 [ 0.         -0.00141421]]
mean_pred: [0. 0.]
cov_pred: [[1. 0.]
 [0. 1.]]


(Array([0., 0.], dtype=float64),
 Array([[1., 0.],
        [0., 1.]], dtype=float64))

# Building UKFPrO

In [34]:
mu, Sigma_raw = unscented_transform(m, P, h_linear)
Sigma = Sigma_raw + R

lambda: -1.999998
S: [[0.00141421 0.        ]
 [0.00028284 0.0017088 ]]
sigma_points: [[1.         2.        ]
 [1.00141421 2.00028284]
 [1.         2.0017088 ]
 [0.99858579 1.99971716]
 [1.         1.9982912 ]]
Wm: [-999998.99997124  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
Wc: [-999995.99997224  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
transformed: [[2.         6.        ]
 [2.00282843 6.00084853]
 [2.         6.0051264 ]
 [1.99717157 5.99915147]
 [2.         5.9948736 ]]
mean_pred: [2. 6.]
cov_pred: [[ 4.   1.2]
 [ 1.2 13.5]]


In [35]:
# ============================================================
# Toy case setup — small, linear h, everything checkable by hand
# ============================================================
d = 2
a = jnp.array([0.5, 1.0])                          # predictive mean
A = jnp.array([[1.2, 0.1], [0.1, 0.9]])            # predictive covariance
H = jnp.array([[2.0, 0.0], [0.0, 3.0]])            # true, linear observation matrix
R = jnp.array([[0.5, 0.0], [0.0, 0.5]])            # observation noise
y = jnp.array([3.0, 5.0])                           # observed value

def h_linear(z):
    return H @ z

L_pred = jnp.linalg.cholesky(A)

In [36]:
# ============================================================
# Building blocks
# ============================================================
def kl_term(m, P, a, A):
    L_pred = jnp.linalg.cholesky(A)
    diff = m - a
    L_diff = jsp_linalg.solve_triangular(L_pred, diff, lower=True)
    mean_term = jnp.sum(jnp.square(L_diff))
    L_P_white = jsp_linalg.solve_triangular(L_pred, P, lower=True)
    L_P_white = jsp_linalg.solve_triangular(L_pred, L_P_white.T, lower=True).T
    trace_term = jnp.trace(L_P_white)
    _, logdet_P = jnp.linalg.slogdet(P)
    logdet_A = 2.0 * jnp.sum(jnp.log(jnp.diagonal(L_pred)))
    return 0.5 * (trace_term + mean_term - m.shape[0] + logdet_A - logdet_P)

def nll_term(m, P, y, h, R):
    mu, Sigma_raw = unscented_transform(m, P, h)
    Sigma = Sigma_raw + R
    diff = y - mu
    L_Sigma = jnp.linalg.cholesky(Sigma)
    L_diff = jsp_linalg.solve_triangular(L_Sigma, diff, lower=True)
    quad = jnp.sum(jnp.square(L_diff))
    logdet_Sigma = 2.0 * jnp.sum(jnp.log(jnp.diagonal(L_Sigma)))
    return 0.5 * (quad + logdet_Sigma + y.shape[0] * jnp.log(2 * jnp.pi))

def exact_linear_obj(m, P, a, A, y, H, R):
    kl = kl_term(m, P, a, A)
    S = H @ P @ H.T + R
    diff = y - H @ m
    nll = 0.5*(diff @ jnp.linalg.solve(S, diff) + jnp.linalg.slogdet(S)[1] + y.shape[0]*jnp.log(2*jnp.pi))
    return kl + nll

def obj_func(params, a, A, y, h, R):
    m_free, U = params
    L_pred = jnp.linalg.cholesky(A)
    B = chol_softplus(U)
    P_candidate = (L_pred @ B) @ (L_pred @ B).T
    return kl_term(m_free, P_candidate, a, A) + nll_term(m_free, P_candidate, y, h, R)

In [37]:
# ============================================================
# Check 1: nll_term matches exact formula, isolated
# ============================================================
mu, Sigma_raw = unscented_transform(a, A, h_linear)
Sigma = Sigma_raw + R
print("Check 1 - mu matches H@a:", jnp.allclose(mu, H @ a, atol=1e-6))
print("Check 1 - Sigma matches H@A@H.T+R:", jnp.allclose(Sigma, H @ A @ H.T + R, atol=1e-6))

lambda: -1.999998
S: [[0.00154919 0.        ]
 [0.0001291  0.00133542]]
sigma_points: [[0.5        1.        ]
 [0.50154919 1.0001291 ]
 [0.5        1.00133542]
 [0.49845081 0.9998709 ]
 [0.5        0.99866458]]
Wm: [-999998.99997124  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
Wc: [-999995.99997224  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
transformed: [[1.         3.        ]
 [1.00309839 3.0003873 ]
 [1.         3.00400625]
 [0.99690161 2.9996127 ]
 [1.         2.99599375]]
mean_pred: [1. 3.]
cov_pred: [[4.8 0.6]
 [0.6 8.1]]
Check 1 - mu matches H@a: True
Check 1 - Sigma matches H@A@H.T+R: True


In [38]:
# ============================================================
# Check 2: obj_func matches exact_linear_obj, at an arbitrary test point
# ============================================================
m_test = jnp.array([0.8, 1.3])
P_test = jnp.array([[0.9, 0.05], [0.05, 0.7]])

val_ut = kl_term(m_test, P_test, a, A) + nll_term(m_test, P_test, y, h_linear, R)
val_exact = exact_linear_obj(m_test, P_test, a, A, y, H, R)
print("Check 2 - obj_func matches exact formula:", jnp.allclose(val_ut, val_exact, atol=1e-6))

lambda: -1.999998
S: [[1.34164079e-03 0.00000000e+00]
 [7.45355993e-05 1.18086597e-03]]
sigma_points: [[0.8        1.3       ]
 [0.80134164 1.30007454]
 [0.8        1.30118087]
 [0.79865836 1.29992546]
 [0.8        1.29881913]]
Wm: [-999998.99997124  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
Wc: [-999995.99997224  249999.99999281  249999.99999281  249999.99999281
  249999.99999281]
transformed: [[1.6        3.9       ]
 [1.60268328 3.90022361]
 [1.6        3.9035426 ]
 [1.59731672 3.89977639]
 [1.6        3.8964574 ]]
mean_pred: [1.6 3.9]
cov_pred: [[3.6 0.3]
 [0.3 6.3]]
Check 2 - obj_func matches exact formula: True


In [39]:
# ============================================================
# Check 3: gradient correctness (finite-difference cross-check)
# ============================================================
params_test = (a, jnp.zeros_like(A))
value, grad = jax.value_and_grad(obj_func)(params_test, a, A, y, h_linear, R)
grad_m, grad_U = grad

eps = 1e-5
m_plus = params_test[0].at[0].add(eps)
value_plus = obj_func((m_plus, params_test[1]), a, A, y, h_linear, R)
numerical_grad_m0 = (value_plus - value) / eps
print("Check 3 - gradient matches finite difference:", jnp.allclose(grad_m[0], numerical_grad_m0, atol=1e-3))

lambda: -1.999998
S: Traced<ShapedArray(float64[2,2])>with<JVPTrace> with
  primal = Array([[0.00154919, 0.        ],
       [0.0001291 , 0.00133542]], dtype=float64)
  tangent = Traced<ShapedArray(float64[2,2])>with<JaxprTrace> with
    pval = (ShapedArray(float64[2,2]), None)
    recipe = JaxprEqnRecipe(eqn_id=<object object at 0x1522a5510>, in_tracers=(Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(bool[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(bool[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>), out_tracer_refs=[<weakref at 0x1523f7510; to 'JaxprTracer' at 0x1523f7560>], out_avals=[ShapedArray(float64[2,2])], primitive=pjit, params={'jaxpr': { lambda ; a:f64[2,2] b:bool[2,2] c:f64[2,2] d:f64[2,2] e:bool[2,2] f:f64[2,2] g:f64[2,2]. let
    h:f64[2,2] = transpose[permutation=(1, 0)] a
    i:f64[2,2] = add a h
    j:f

In [40]:
# ============================================================
# Check 4: run the optimiser, check convergence stability
# ============================================================
def solve_update(a, A, y, h, R, n_iters=30):
    params = (a, jnp.zeros_like(A))
    def loss_fn(p):
        return obj_func(p, a, A, y, h, R)
    solver = optax.lbfgs(memory_size=10, scale_init_precond=True)
    state = solver.init(params)
    for i in range(n_iters):
        val, g = jax.value_and_grad(loss_fn)(params)
        updates, state = solver.update(g, state, params, value=val, grad=g, value_fn=loss_fn)
        params = optax.apply_updates(params, updates)
    return params, loss_fn(params)

(m_final, U_final), val_30 = solve_update(a, A, y, h_linear, R, n_iters=30)
(_, _), val_100 = solve_update(a, A, y, h_linear, R, n_iters=100)
print("Check 4 - value stable across iteration budgets:", jnp.allclose(val_30, val_100, atol=1e-6))

B_final = chol_softplus(U_final)
P_final = (L_pred @ B_final) @ (L_pred @ B_final).T
print("m_final:", m_final)
print("Final objective value:", val_30)

lambda: -1.999998
S: Traced<ShapedArray(float64[2,2])>with<JVPTrace> with
  primal = Array([[0.00154919, 0.        ],
       [0.0001291 , 0.00133542]], dtype=float64)
  tangent = Traced<ShapedArray(float64[2,2])>with<JaxprTrace> with
    pval = (ShapedArray(float64[2,2]), None)
    recipe = JaxprEqnRecipe(eqn_id=<object object at 0x1522a5520>, in_tracers=(Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(bool[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(bool[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>, Traced<ShapedArray(float64[2,2]):JaxprTrace>), out_tracer_refs=[<weakref at 0x1523f9d50; to 'JaxprTracer' at 0x1523f9ee0>], out_avals=[ShapedArray(float64[2,2])], primitive=pjit, params={'jaxpr': { lambda ; a:f64[2,2] b:bool[2,2] c:f64[2,2] d:f64[2,2] e:bool[2,2] f:f64[2,2] g:f64[2,2]. let
    h:f64[2,2] = transpose[permutation=(1, 0)] a
    i:f64[2,2] = add a h
    j:f

# Linear toy example - UKFPrO vs PrO

In [1]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import jax.scipy.linalg as jsp_linalg
import optax
from profilter import chol_softplus

# --- Shared toy setup ---
a = jnp.array([0.5, 1.0])
A = jnp.array([[1.2, 0.1], [0.1, 0.9]])
H = jnp.array([[2.0, 0.0], [0.0, 3.0]])
R = jnp.array([[0.5, 0.0], [0.0, 0.5]])
y = jnp.array([3.0, 5.0])
L_pred = jnp.linalg.cholesky(A)
d = a.shape[0]
active_mask = jnp.tril(jnp.ones_like(A))
n_active = jnp.sum(active_mask)


# ============================================================
# Strategy 1: Profiled mean, outer/inner loop (PrOFilter's approach)
# ============================================================
def solve_profiled_exact(a, A, y, H, R, n_outer=10, n_inner=30):
    HAHt = H @ A @ H.T
    m_lin = a

    for outer_i in range(n_outer):
        y_hat = H @ m_lin
        predictive_error = y - y_hat - H @ (a - m_lin)

        def obj_func(U):
            B = chol_softplus(U)
            L_candidate = L_pred @ B
            P = L_candidate @ L_candidate.T
            S = H @ P @ H.T + R

            mean_delta = A @ H.T @ jnp.linalg.solve(S + HAHt, predictive_error)
            mean_new = a + mean_delta

            residual_y = y - y_hat - H @ (mean_new - m_lin)
            chol_S = jnp.linalg.cholesky(S)
            whitened_residual_y = jsp_linalg.solve_triangular(chol_S, residual_y, lower=True)
            predictive_quadratic = 0.5 * jnp.sum(jnp.square(whitened_residual_y))

            determinant_term = (
                jnp.sum(jnp.log(jnp.diagonal(chol_S))) - jnp.sum(jnp.log(jnp.diagonal(B)))
            )

            residual_mean = mean_new - a
            whitened_residual_mean = jsp_linalg.solve_triangular(L_pred, residual_mean, lower=True)
            mean_kl = 0.5 * jnp.sum(jnp.square(whitened_residual_mean))

            covariance_kl = 0.5 * jnp.sum(jnp.square(B))

            value = predictive_quadratic + determinant_term + mean_kl + covariance_kl
            return value, mean_new

        value_and_grad_fn = jax.value_and_grad(obj_func, has_aux=True)
        solver = optax.lbfgs(memory_size=10, scale_init_precond=True)

        U = jnp.zeros_like(A)
        state = solver.init(U)
        (value, mean_new), grad = value_and_grad_fn(U)

        for inner_i in range(n_inner):
            def value_fn(U0):
                v, _ = obj_func(U0)
                return v
            updates, state = solver.update(grad, state, U, value=value, grad=grad, value_fn=value_fn)
            U = optax.apply_updates(U, updates)
            (value, mean_new), grad = value_and_grad_fn(U)

        m_lin = mean_new

    B_final = chol_softplus(U)
    P_final = (L_pred @ B_final) @ (L_pred @ B_final).T
    return mean_new, P_final, value


# ============================================================
# Strategy 2: Joint (m,P) optimisation, single loop
# ============================================================
def solve_joint_exact(a, A, y, H, R, n_iters=500):
    def kl_term(m, P):
        diff = m - a
        L_diff = jsp_linalg.solve_triangular(L_pred, diff, lower=True)
        mean_term = jnp.sum(jnp.square(L_diff))
        L_P_white = jsp_linalg.solve_triangular(L_pred, P, lower=True)
        L_P_white = jsp_linalg.solve_triangular(L_pred, L_P_white.T, lower=True).T
        trace_term = jnp.trace(L_P_white)
        _, logdet_P = jnp.linalg.slogdet(P)
        logdet_A = 2.0 * jnp.sum(jnp.log(jnp.diagonal(L_pred)))
        return 0.5 * (trace_term + mean_term - d + logdet_A - logdet_P)

    def nll_term(m, P):
        S = H @ P @ H.T + R
        diff = y - H @ m
        chol_S = jnp.linalg.cholesky(S)
        whitened = jsp_linalg.solve_triangular(chol_S, diff, lower=True)
        quad = jnp.sum(jnp.square(whitened))
        logdet_S = 2.0 * jnp.sum(jnp.log(jnp.diagonal(chol_S)))
        return 0.5 * (quad + logdet_S + y.shape[0] * jnp.log(2 * jnp.pi))

    def obj_func(params):
        m_free, U = params
        B = chol_softplus(U)
        P_candidate = (L_pred @ B) @ (L_pred @ B).T
        return kl_term(m_free, P_candidate) + nll_term(m_free, P_candidate)

    value_and_grad_fn = jax.value_and_grad(obj_func)
    solver = optax.lbfgs(memory_size=10, scale_init_precond=True)
    params = (a, jnp.zeros_like(A))
    state = solver.init(params)

    def value_fn(p):
        return obj_func(p)

    value, grad = value_and_grad_fn(params)
    for i in range(n_iters):
        updates, state = solver.update(grad, state, params, value=value, grad=grad, value_fn=value_fn)
        params = optax.apply_updates(params, updates)
        value, grad = value_and_grad_fn(params)

    m_final, U_final = params
    B_final = chol_softplus(U_final)
    P_final = (L_pred @ B_final) @ (L_pred @ B_final).T
    return m_final, P_final, value


# ============================================================
# Run and compare
# ============================================================
m_profiled, P_profiled, val_profiled = solve_profiled_exact(a, A, y, H, R)
m_joint, P_joint, val_joint = solve_joint_exact(a, A, y, H, R)

print("Profiled (outer/inner) mean:", m_profiled)
print("Joint mean:", m_joint)
print("Mean diff:", jnp.abs(m_profiled - m_joint))
print()
print("Profiled value:", val_profiled)
print("Joint value:", val_joint)
print("Value diff:", jnp.abs(val_profiled - val_joint))

Profiled (outer/inner) mean: [1.16266893 1.47647333]
Joint mean: [1.16266893 1.47647333]
Mean diff: [0.00000000e+00 2.22044605e-16]

Profiled value: 2.7970666587606283
Joint value: 3.6349437251699737
Value diff: 0.8378770664093453


In [2]:
print("P_profiled:", P_profiled)
print("P_joint:", P_joint)
print("Diff:", jnp.abs(P_profiled - P_joint))

P_profiled: [[0.44873834 0.09120185]
 [0.09120185 0.24802327]]
P_joint: [[0.44873834 0.09120185]
 [0.09120185 0.24802327]]
Diff: [[0.00000000e+00 5.55111512e-17]
 [5.55111512e-17 2.77555756e-17]]
